# Argument and research validation
Synthetic cards exercise the same parser, index builder, research context and query port used by the CLI.

In [ ]:
import sys, tempfile, shutil
from pathlib import Path
root = Path.cwd()
while not (root / "knowledge_palace").is_dir():
    root = root.parent
sys.path.insert(0, str(root))
from knowledge_palace.tests.test_research import ARGUMENT, SYNTHESIS, MINI
from knowledge_palace.graph.builder import write_index, ensure_index
from knowledge_palace.graph.port import GraphQueryPort
from knowledge_palace.interaction.research_helpers import research_context, render_progress
temp = tempfile.TemporaryDirectory()
vault, state = Path(temp.name)/"vault", Path(temp.name)/"state"
shutil.copytree(MINI, vault)
paper = vault/"papers/alpha-2020-echo.md"
paper.write_text(paper.read_text(encoding="utf-8") + ARGUMENT, encoding="utf-8")
brief = vault / "briefs/2026-07-13-gaps.md"
brief.write_text(brief.read_text(encoding="utf-8") + SYNTHESIS, encoding="utf-8")
_, document = write_index(vault, state)
document["payload"]["identity_report"]

In [ ]:
context = research_context(document["payload"], "rare-window-token")
context["works"], context["relations"]

In [ ]:
response = GraphQueryPort(vault, state).query_context("claim:alpha-2020-echo#C1")
assert response["node"]["id"] == "claim:alpha-2020-echo#C1"
assert context["relations"][0]["attrs"]["claim_ref"] == "claim:alpha-2020-echo#C1"
print(render_progress(context))


In [ ]:
from knowledge_palace.semantic.update_helpers import load_updates
paper.write_text(paper.read_text(encoding="utf-8").replace("rare-window-token", "changed-window-token"), encoding="utf-8")
write_index(vault, state)
registry = vault / "concepts.md"
registry.write_text(registry.read_text(encoding="utf-8").replace("| echo cancel |", "| echo cancel, acoustic |"), encoding="utf-8")
write_index(vault, state)
reviews = [row for row in load_updates(state)["entries"] if row["node_id"].startswith("brief:")]
assert reviews[0]["status"] == "pending"
reviews[0]["status"], reviews[0]["events"]


In [ ]:
temp.cleanup()